
### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

In [9]:
from langchain_groq import ChatGroq

model=ChatGroq(model='openai/gpt-oss-20b')

In [10]:
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000245D7BBB2D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000245D7BB9290>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [11]:
from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The year the movie was released")
    genre:str=Field(...,description="The genre of the movie")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The rating of the movie")

In [12]:
model_with_str=model.with_structured_output(Movie)
model_with_str

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000245D7BBB2D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000245D7BB9290>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The tit

In [13]:
model.invoke("Provide details about the moview Inception")

AIMessage(content="**Inception (2010)**  \n\n| Category | Details |\n|----------|---------|\n| **Director** | Christopher Nolan |\n| **Screenwriter** | Christopher Nolan |\n| **Producer(s)** | Emma Thomas, Christopher Nolan, Christopher McQuarrie |\n| **Studio** | Warner Bros. Pictures, Syncopy |\n| **Distributor** | Warner Bros. Pictures |\n| **Release Date** | 16 July 2010 (USA) |\n| **Runtime** | 148 minutes |\n| **Budget** | ~US$160\u202fmillion |\n| **Box‑office** | ~US$829\u202fmillion worldwide |\n| **Genre** | Science‑fiction, Thriller, Action |\n| **Language** | English |\n| **Country** | United States |\n\n---\n\n### Main Cast\n\n| Actor | Role |\n|-------|------|\n| Leonardo DiCaprio | Dom Cobb – extraction specialist |\n| Joseph Gordon‑Levitt | Arthur – point‑man & lock‑pick specialist |\n| Ellen Page | Ariadne – architect of dream layers |\n| Tom Hardy | Eames – forger & con artist |\n| Ken Watanabe | Saito – corporate client |\n| Cillian Murphy | Robert Fischer – target’s

In [15]:
response=model_with_str.invoke("Provide details about the moview Inception")
response

Movie(title='Inception', year=2010, genre='Science Fiction', director='Christopher Nolan', rating=8.8)

### Nested Structure

In [16]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response


MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Sci-Fi', 'Thriller'], budget=160000000.0)